# Llama 3.2 1B Fine-Tuning with Pre-computed Token Embeddings

This notebook fine-tunes Llama 3.2 1B using pre-computed token embeddings instead of token IDs.
The approach:
1. Extract the embedding layer from Llama 3.2 1B
2. Pre-compute embeddings for all tokens in the dataset and save to disk
3. LoRA fine-tune the model to accept embeddings directly via `inputs_embeds`
4. Save the trained model

In [1]:
# Load HuggingFace token from .env file
from dotenv import load_dotenv
load_dotenv()

import os
from huggingface_hub import HfApi, login
import json

# Set HF cache FIRST
os.environ['HF_HOME'] = '/lus/lfs1aip2/home/s5e/jrosser.s5e/huggingface'
os.environ['HUGGINGFACE_HUB_CACHE'] = '/lus/lfs1aip2/home/s5e/jrosser.s5e/huggingface'

import torch
from datasets import load_dataset, Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    pipeline,
    logging,
)
from peft import LoraConfig, PeftModel, get_peft_model
from tqdm import tqdm
import numpy as np

Skipping import of cpp extensions due to incompatible torch version 2.7.0+cu128 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info


## Configuration

In [2]:
# Model configuration
model_name = "meta-llama/Llama-3.2-1B-Instruct"
dataset_name = "rk404/recipe_short"

# Output paths
embeddings_dir = "/lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/embeddings"
new_model = "/lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/llama-3.2-1B-embeddings-finetune"
output_dir = "/lus/lfs1aip2/home/s5e/jrosser.s5e/llama_3/results_embeddings"

# Create embeddings directory
os.makedirs(embeddings_dir, exist_ok=True)

In [3]:
# LoRA parameters
lora_r = 8
lora_alpha = 16
lora_dropout = 0.1

# bitsandbytes parameters
use_4bit = True
bnb_4bit_compute_dtype = "float16"
bnb_4bit_quant_type = "nf4"
use_nested_quant = False

# Training parameters
num_train_epochs = 10
per_device_train_batch_size = 4
gradient_accumulation_steps = 1
learning_rate = 2e-4
weight_decay = 0.001
max_grad_norm = 0.3
warmup_ratio = 0.03
max_seq_length = 2048

# Device
device_map = {"" : 0}

## Step 1: Load Model and Extract Embedding Layer

In [4]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Tokenizer loaded. Vocab size: {tokenizer.vocab_size}")

Tokenizer loaded. Vocab size: 128000


In [5]:
# Configure bitsandbytes for 4-bit quantization
compute_dtype = getattr(torch, bnb_4bit_compute_dtype)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=use_4bit,
    bnb_4bit_quant_type=bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=use_nested_quant,
)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map=device_map
)
model.config.use_cache = False
model.config.pretraining_tp = 1

print(f"Model loaded on device: {model.device}")

Model loaded on device: cuda:0


In [6]:
# Extract embedding layer
embed_layer = model.get_input_embeddings()
embed_weights = embed_layer.weight

print(f"Embedding layer shape: {embed_weights.shape}")
print(f"Embedding dtype: {embed_weights.dtype}")
print(f"Hidden size: {model.config.hidden_size}")

Embedding layer shape: torch.Size([128256, 2048])
Embedding dtype: torch.float16
Hidden size: 2048


## Step 2: Prepare Dataset

In [7]:
# Load dataset
dataset_subset = load_dataset(dataset_name, split="train")
dataset_subset = dataset_subset.select(range(1_000))

messages_list = []
skipped_long = 0
skipped_error = 0

for row in dataset_subset:
    try:
        # directions are stored as a Python-list string -> make a list
        directions_list = eval(row["directions"])
        directions_text = "\n".join(d.strip() for d in directions_list if d.strip())

        # skip very short / broken recipes
        if len(directions_text) < 50:
            continue

        ingredients = eval(row["ingredients"])
        if not ingredients:
            continue

        # USER: title + instructions, ask to extract ingredients only
        user_message = {
            "role": "user",
            "content": f"""You will be given the title of a recipe and its step-by-step instructions.
Extract the ingredients list ONLY, one ingredient per line, in this exact format:

Ingredients:
* ingredient 1
* ingredient 2
END

Title: {row['title']}

Instructions:
{directions_text}
"""
        }

        # ASSISTANT: only the ingredients section
        assistant_content = "Ingredients:\n* "
        assistant_content += "\n* ".join(ingredients)
        assistant_content += "\nEND"

        assistant_message = {
            "role": "assistant",
            "content": assistant_content
        }

        chat_text = tokenizer.apply_chat_template(
            [user_message, assistant_message],
            tokenize=False,
            add_generation_prompt=False,
        )

        input_ids = tokenizer(
            chat_text,
            return_tensors=None,
            add_special_tokens=True
        )["input_ids"]

        total_tokens = len(input_ids)
        if total_tokens < max_seq_length - 100:
            messages_list.append([user_message, assistant_message])
        else:
            skipped_long += 1

    except Exception as e:
        skipped_error += 1

print(f"Subset loaded: {len(dataset_subset)} examples")
print(f"Skipped (too long): {skipped_long}")
print(f"Skipped (errors): {skipped_error}")
print(f"Final dataset size: {len(messages_list)} examples")

Subset loaded: 1000 examples
Skipped (too long): 0
Skipped (errors): 0
Final dataset size: 968 examples


## Step 3: Pre-compute and Save Embeddings

In [8]:
# Check if embeddings already exist
embeddings_file = os.path.join(embeddings_dir, "recipe_embeddings.pt")

if os.path.exists(embeddings_file):
    print(f"Loading existing embeddings from {embeddings_file}")
    saved_data = torch.load(embeddings_file)
    all_embeddings = saved_data['embeddings']
    all_attention_masks = saved_data['attention_masks']
    all_labels = saved_data['labels']
    print(f"Loaded {len(all_embeddings)} pre-computed embeddings")
else:
    print("Pre-computing embeddings...")
    
    all_embeddings = []
    all_attention_masks = []
    all_labels = []
    
    with torch.no_grad():
        for i, messages in enumerate(tqdm(messages_list, desc="Computing embeddings")):
            # Tokenize the full conversation
            chat_text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False,
            )
            
            encoded = tokenizer(
                chat_text,
                return_tensors="pt",
                add_special_tokens=True,
                padding=False,
                truncation=True,
                max_length=max_seq_length
            )
            
            input_ids = encoded["input_ids"].to(model.device)
            attention_mask = encoded["attention_mask"]
            
            # Get embeddings from the embedding layer
            embeddings = embed_layer(input_ids)  # [1, seq_len, hidden_size]
            
            # For causal LM, labels are the same as input_ids (shifted internally)
            labels = input_ids.clone()
            
            # Store on CPU to save GPU memory
            all_embeddings.append(embeddings.cpu().squeeze(0))  # [seq_len, hidden_size]
            all_attention_masks.append(attention_mask.squeeze(0))  # [seq_len]
            all_labels.append(labels.cpu().squeeze(0))  # [seq_len]
    
    # Save to disk
    print(f"Saving embeddings to {embeddings_file}")
    torch.save({
        'embeddings': all_embeddings,
        'attention_masks': all_attention_masks,
        'labels': all_labels,
    }, embeddings_file)
    print(f"Saved {len(all_embeddings)} embeddings to disk")

Loading existing embeddings from /lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/embeddings/recipe_embeddings.pt
Loaded 968 pre-computed embeddings


In [9]:
# Verify embeddings
print(f"Number of examples: {len(all_embeddings)}")
print(f"First embedding shape: {all_embeddings[0].shape}")
print(f"First attention mask shape: {all_attention_masks[0].shape}")
print(f"First labels shape: {all_labels[0].shape}")
print(f"Embedding dtype: {all_embeddings[0].dtype}")

Number of examples: 968
First embedding shape: torch.Size([251, 2048])
First attention mask shape: torch.Size([251])
First labels shape: torch.Size([251])
Embedding dtype: torch.float16


## Step 4: Custom Dataset and Data Collator

The HuggingFace model natively supports `inputs_embeds` - when you pass embeddings instead of `input_ids`, it skips the embedding layer automatically. No custom wrapper needed!

In [10]:
from torch.utils.data import Dataset as TorchDataset

class EmbeddingDataset(TorchDataset):
    """Dataset that returns pre-computed embeddings instead of token IDs."""
    
    def __init__(self, embeddings, attention_masks, labels):
        self.embeddings = embeddings
        self.attention_masks = attention_masks
        self.labels = labels
    
    def __len__(self):
        return len(self.embeddings)
    
    def __getitem__(self, idx):
        return {
            'inputs_embeds': self.embeddings[idx],
            'attention_mask': self.attention_masks[idx],
            'labels': self.labels[idx],
        }


class EmbeddingDataCollator:
    """Collator that pads embeddings, attention masks, and labels to the same length."""
    
    def __init__(self, pad_token_id, hidden_size):
        self.pad_token_id = pad_token_id
        self.hidden_size = hidden_size
    
    def __call__(self, features):
        # Find max length in batch
        max_len = max(f['inputs_embeds'].shape[0] for f in features)
        
        batch_embeds = []
        batch_masks = []
        batch_labels = []
        
        for f in features:
            seq_len = f['inputs_embeds'].shape[0]
            pad_len = max_len - seq_len
            
            # Pad embeddings with zeros
            if pad_len > 0:
                embed_pad = torch.zeros(pad_len, self.hidden_size, dtype=f['inputs_embeds'].dtype)
                padded_embeds = torch.cat([f['inputs_embeds'], embed_pad], dim=0)
            else:
                padded_embeds = f['inputs_embeds']
            
            # Pad attention mask with zeros
            if pad_len > 0:
                mask_pad = torch.zeros(pad_len, dtype=f['attention_mask'].dtype)
                padded_mask = torch.cat([f['attention_mask'], mask_pad], dim=0)
            else:
                padded_mask = f['attention_mask']
            
            # Pad labels with -100 (ignored in loss)
            if pad_len > 0:
                label_pad = torch.full((pad_len,), -100, dtype=f['labels'].dtype)
                padded_labels = torch.cat([f['labels'], label_pad], dim=0)
            else:
                padded_labels = f['labels']
            
            batch_embeds.append(padded_embeds)
            batch_masks.append(padded_mask)
            batch_labels.append(padded_labels)
        
        return {
            'inputs_embeds': torch.stack(batch_embeds),
            'attention_mask': torch.stack(batch_masks),
            'labels': torch.stack(batch_labels),
        }

print("Dataset and collator classes defined")

Dataset and collator classes defined


In [11]:
# Create dataset and collator
train_dataset = EmbeddingDataset(all_embeddings, all_attention_masks, all_labels)
data_collator = EmbeddingDataCollator(
    pad_token_id=tokenizer.pad_token_id,
    hidden_size=model.config.hidden_size
)

print(f"Training dataset size: {len(train_dataset)}")

# Test the collator
sample_batch = data_collator([train_dataset[0], train_dataset[1]])
print(f"Sample batch inputs_embeds shape: {sample_batch['inputs_embeds'].shape}")
print(f"Sample batch attention_mask shape: {sample_batch['attention_mask'].shape}")
print(f"Sample batch labels shape: {sample_batch['labels'].shape}")

Training dataset size: 968
Sample batch inputs_embeds shape: torch.Size([2, 251, 2048])
Sample batch attention_mask shape: torch.Size([2, 251])
Sample batch labels shape: torch.Size([2, 251])


## Step 5: Configure LoRA

Apply LoRA to the model. When we pass `inputs_embeds` to the model, it automatically skips the embedding layer - no custom wrapper needed!

In [12]:
# Configure LoRA - targeting transformer layers, NOT the embedding layer
peft_config = LoraConfig(
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    r=lora_r,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "down_proj"]  # Same as reference notebook
)

# Apply LoRA to the model
model = get_peft_model(model, peft_config)

# Print trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable_params:,}")
print(f"All parameters: {all_params:,}")
print(f"Trainable %: {100 * trainable_params / all_params:.4f}%")

print("\nModel will accept inputs_embeds directly (embedding layer bypassed)")

Trainable parameters: 2,162,688
All parameters: 751,437,824
Trainable %: 0.2878%

Model will accept inputs_embeds directly (embedding layer bypassed)


## Step 7: Training

In [13]:
from transformers import TrainerCallback

class SavePerEpochCallback(TrainerCallback):
    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        epoch = int(state.epoch)
        save_path = f"{new_model}_{epoch}"
        model.save_pretrained(save_path)
        tokenizer.save_pretrained(save_path)
        print(f"Saved: {save_path}")

save_callback = SavePerEpochCallback()

In [14]:
# Training arguments
training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    optim="paged_adamw_32bit",
    save_steps=100,
    logging_steps=25,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    fp16=False,
    bf16=True,
    max_grad_norm=max_grad_norm,
    warmup_ratio=warmup_ratio,
    group_by_length=False,  # Can't use with variable-length embeddings
    lr_scheduler_type="constant",
    report_to="tensorboard",
    remove_unused_columns=False,  # Important: keep our custom columns
)

In [15]:
# Create trainer - model will receive inputs_embeds from our collator
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
    callbacks=[save_callback],
)

print("Trainer created")
print(f"Model type: {type(trainer.model).__name__}")
print("When inputs_embeds is passed, the embedding layer is automatically bypassed!")

Trainer created
Model type: PeftModelForCausalLM
When inputs_embeds is passed, the embedding layer is automatically bypassed!


In [16]:
# Train!
trainer.train()

# Save final model
trainer.model.save_pretrained(new_model)
tokenizer.save_pretrained(new_model)
print(f"LoRA weights and tokenizer saved to: {new_model}")

Could not estimate the number of tokens of the input, floating-point operations will not be computed


Step,Training Loss
25,1.944800
50,1.169100
75,1.127900
100,1.059700
125,0.985100
150,1.000400
175,0.991000
200,0.920800
225,0.938700
250,0.965900


Saved: /lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/llama-3.2-1B-embeddings-finetune_1
Saved: /lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/llama-3.2-1B-embeddings-finetune_2
Saved: /lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/llama-3.2-1B-embeddings-finetune_3
Saved: /lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/llama-3.2-1B-embeddings-finetune_4
Saved: /lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/llama-3.2-1B-embeddings-finetune_5
Saved: /lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/llama-3.2-1B-embeddings-finetune_6
Saved: /lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/llama-3.2-1B-embeddings-finetune_7
Saved: /lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/llama-3.2-1B-embeddings-finetune_8
Saved: /lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/llama-3.2-1B-embeddings-finetune_9
Saved: /lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/llama-3.2-1B-embeddings-finetune_10
LoRA weights and tokenizer saved to: /lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/llama-3.2-1B-embeddings-finetune


In [17]:
# Verify the trained model structure
print("Trained model:")
print(model)

Trained model:
PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 2048)
        (layers): ModuleList(
          (0-15): 16 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (

## Step 8: Test the Trained Model

Test the model with embeddings as input to verify it works correctly.

In [ ]:
print("Testing merged model with embeddings input (generate 500 tokens):")
print("=" * 80)

# Get a test example
test_messages = messages_list[0]
user_msg = test_messages[0]

# Tokenize and get embeddings
prompt = tokenizer.apply_chat_template([user_msg], tokenize=False, add_generation_prompt=True)
encoded = tokenizer(prompt, return_tensors="pt", add_special_tokens=True)
input_ids = encoded["input_ids"].to(model.device)

# Get embeddings from the embedding layer
merged_embed_layer = model.get_input_embeddings()
with torch.no_grad():
    input_embeds = merged_embed_layer(input_ids)

print(f"Input embeddings shape: {input_embeds.shape}")

# Generate continuation using model.generate and embeddings
# NOTE: Early stopping and max_new_tokens are set to 500
gen_attention_mask = encoded["attention_mask"].to(model.device)

with torch.no_grad():
    generated_outputs = model.generate(
        inputs_embeds=input_embeds,
        attention_mask=gen_attention_mask,
        max_new_tokens=500,
        do_sample=False,     # Deterministic generation (set True if you want sampling)
        num_beams=1,         # No beam search – greedy decoding
        pad_token_id=tokenizer.eos_token_id,
    )

# Print results
decoded_generated = tokenizer.batch_decode(generated_outputs, skip_special_tokens=True)
print("Decoded generated output (up to 500 tokens):")
for i, txt in enumerate(decoded_generated):
    print(f"[Sample {i}]: {txt}")



Testing merged model with embeddings input (generate 500 tokens):
Input embeddings shape: torch.Size([1, 178, 2048])
Decoded generated output (up to 500 tokens):
[Sample 0]: Ingredients:
* 2 c. self-rising flour
* 2 c. brown sugar, packed
* 1/2 c. evaporated milk
* 1/2 tsp. vanilla
* 1/2 c. chopped nuts
* 3 Tbsp. butter or margarine, melted
* 3/4 c. corn flake cereal, rinsed in cold water, drained
END


In [ ]:
# Test generation using the standard pipeline
print("\nGenerating with standard pipeline:")
print("=" * 80)

pipe = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    max_length=512,
    do_sample=False,
    num_beams=1,
    return_full_text=False,
)

result = pipe(prompt)
print(f"Prompt: {user_msg['content'][:200]}...")
print(f"\nResponse:\n{result[0]['generated_text']}")


Generating with pipeline using input_ids directly:
Pipeline does not support input_ids directly: can only concatenate str (not "dict") to str
Falling back to passing text prompt as input.
Prompt: You will be given the title of a recipe and its step-by-step instructions.
Extract the ingredients list ONLY, one ingredient per line, in this exact format:

Ingredients:
* ingredient 1
* ingredient 2...

Response:
Ingredients:
* 2 c. self-rising flour
* 1 1/2 c. brown sugar, packed
* 1/2 c. evaporated milk
* 1/2 tsp. vanilla
* 1/2 c. chopped nuts
* 1/2 c. butter or margarine, melted
* 3/4 c. uncooked Rice Chex
END
